# 🤟 ISL-CSLRT: EMF-SLT + ST-GCN Encoder-Decoder

## Based on Two Research Papers:
| Paper | Contribution Used |
|---|---|
| **Sharma et al. (COMSNETS 2025)** — ISL Encoder-Decoder | MediaPipe keypoints → **ST-GCN** spatial-temporal graph features → Encoder-Decoder |
| **Hu et al. (ICASSP 2024)** — EMF-SLT | **Vector Quantizer** for sign-gloss alignment + **Multi-modal Fusion** + **KL mutual learning** |

## Combined Architecture:
```
Keypoints (225D) → ST-GCN Encoder → Sign Features S
Gloss IDs        → Embedding      → Gloss Features G
                                    ↓
              Vector Quantizer (align S→G in discrete space)
              Fusion Module (cross-attention: [S; S'] × S)
                                    ↓
              Text Decoder (Transformer)
              Multi-task: S2T loss + G2T loss + KL divergence
              Total Loss = NLL + α·KL + β·Align
```
**Evaluation:** BLEU-1/2/3/4 + ROUGE (as used in Paper 2)

In [1]:
import os, re, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#e6edf3','axes.labelcolor':'#e6edf3','axes.titlecolor':'#58a6ff',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','grid.color':'#21262d','font.size':11})

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE        = r'D:\ISL2\ISL_CSLRT_Corpus'
DATA_DIR    = os.path.join(BASE, 'preprocessed_data')  # from Phase1 (keypoints)
VOCAB_PATH  = os.path.join(BASE, 'gloss_vocab.json')
CKPT_PATH   = os.path.join(BASE, 'best_emf_slt.pth')

# Special tokens for seq2seq decoder
BOS, EOS, PAD = '<bos>', '<eos>', '<pad>'

torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


---
## Phase 1 — Vocabulary (shared gloss+text, per Paper 2)

In [2]:
def tokenize(text):
    if not text or pd.isna(text): return []
    return [w for w in re.sub(r"[^a-z0-9 ']", ' ', str(text).lower().strip()).split() if w]

# Collect all tokens from labels
all_tokens = []
for split in ['train','val','test']:
    d = os.path.join(DATA_DIR, split)
    if not os.path.isdir(d): continue
    for f in os.listdir(d):
        if f.endswith('_label.txt'):
            all_tokens.extend(tokenize(open(os.path.join(d,f),encoding='utf-8').read()))

# Build shared vocab with special tokens (Paper 2: shared gloss+text vocabulary)
unique = sorted(set(all_tokens))
vocab = {PAD:0, BOS:1, EOS:2, '<blank>':3}
for i, t in enumerate(unique, start=4): vocab[t] = i
idx2tok = {v:k for k,v in vocab.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX = vocab[PAD]; BOS_IDX = vocab[BOS]; EOS_IDX = vocab[EOS]; BLANK_IDX = vocab['<blank>']

json.dump(vocab, open(VOCAB_PATH,'w'), indent=2)
print(f'Vocabulary size   : {VOCAB_SIZE}  (incl. PAD/BOS/EOS/blank)')
print(f'Random CTC base   : log({VOCAB_SIZE}) = {math.log(VOCAB_SIZE):.3f}')
print(f'Sample tokens     : {unique[:15]}')

Vocabulary size   : 184  (incl. PAD/BOS/EOS/blank)
Random CTC base   : log(184) = 5.215
Sample tokens     : ['a', 'about', 'abuse', 'afraid', 'agree', 'all', 'am', 'and', 'angry', 'any', 'anything', 'appreciate', 'are', 'bad', 'be']


---
## Phase 2 — Dataset
> Uses MediaPipe keypoints (225D, hands+pose) from `preprocessed_data/`.
> Checks if v1 (1629D) or v2 (225D) features exist.

In [3]:
class ISLSeq2SeqDataset(Dataset):
    """
    Loads keypoint sequences as encoder input.
    Gloss token IDs used as both CTC targets and decoder inputs.
    """
    def __init__(self, split_dir, vocab, max_src=400, max_tgt=30, training=False):
        self.samples=[]; self.max_src=max_src; self.max_tgt=max_tgt; self.training=training
        self.feat_dim = None
        for f in sorted(os.listdir(split_dir)):
            if not f.endswith('.npy'): continue
            lb = os.path.join(split_dir, f[:-4]+'_label.txt')
            if not os.path.exists(lb): continue
            txt = open(lb, encoding='utf-8').read().strip()
            toks= tokenize(txt)
            if not toks: continue
            gloss_ids = [vocab[t] for t in toks if t in vocab]
            tgt_ids   = [BOS_IDX] + gloss_ids + [EOS_IDX]
            self.samples.append((os.path.join(split_dir,f), gloss_ids, tgt_ids, txt))
        if self.samples:
            self.feat_dim = np.load(self.samples[0][0]).shape[1]
        print(f'  {split_dir.split(os.sep)[-1]:6s}: {len(self.samples)} samples  feat_dim={self.feat_dim}')

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        np_, gloss_ids, tgt_ids, _ = self.samples[idx]
        x = np.nan_to_num(np.load(np_).astype(np.float32))
        # Augmentation (Paper 1: helps with signer variation)
        if self.training:
            if random.random()<0.5: x += np.random.randn(*x.shape).astype(np.float32)*0.008
            if random.random()<0.4:
                T=x.shape[0]; r=random.uniform(0.85,1.15)
                i=np.round(np.linspace(0,T-1,max(int(T*r),4))).astype(int).clip(0,T-1); x=x[i]
        if x.shape[0] > self.max_src:
            i=np.round(np.linspace(0,x.shape[0]-1,self.max_src)).astype(int); x=x[i]
        tgt = tgt_ids[:self.max_tgt]
        return (torch.tensor(x,dtype=torch.float32),
                torch.tensor(gloss_ids,dtype=torch.long),
                torch.tensor(tgt,dtype=torch.long))

def collate(batch):
    xs, gs, ts = zip(*batch)
    xl = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    gl = torch.tensor([g.shape[0] for g in gs], dtype=torch.long)
    xs_pad = pad_sequence(xs, batch_first=True)
    gs_pad = pad_sequence(gs, batch_first=True, padding_value=PAD_IDX)
    ts_pad = pad_sequence(ts, batch_first=True, padding_value=PAD_IDX)
    return xs_pad, gs_pad, ts_pad, xl, gl

# Try preprocessed_v2 (225D) first, else fall back to preprocessed_data (1629D)
for dd in ['preprocessed_v2', 'preprocessed_data']:
    candidate = os.path.join(BASE, dd)
    if os.path.isdir(os.path.join(candidate,'train')):
        DATA_DIR = candidate; break
print(f'Using: {DATA_DIR}')

train_ds = ISLSeq2SeqDataset(os.path.join(DATA_DIR,'train'), vocab, training=True)
val_ds   = ISLSeq2SeqDataset(os.path.join(DATA_DIR,'val'),   vocab, training=False)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=collate, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, collate_fn=collate, num_workers=0)

INPUT_DIM = train_ds.feat_dim
print(f'Input dim: {INPUT_DIM} | Vocab: {VOCAB_SIZE}')

Using: D:\ISL2\ISL_CSLRT_Corpus\preprocessed_v2
  train : 274 samples  feat_dim=225
  val   : 0 samples  feat_dim=None
Input dim: 225 | Vocab: 184


---
## Phase 3 — ST-GCN Sign Encoder (Paper 1)
> **From Sharma et al. (COMSNETS 2025):** Graph Convolutional Network captures spatial relationships between joints. Combined with temporal convolution for ST-GCN = Spatial-Temporal Graph Convolution.

In [4]:
class STGCNBlock(nn.Module):
    """
    Spatial-Temporal Graph Convolution Block. (Paper 1: Sharma et al.)
    H^(l+1) = σ(D̃^(-1/2) Ã D̃^(-1/2) H^(l) W^(l))  — Eq(1) from Paper 1
    Combined with temporal 1D convolution.
    """
    def __init__(self, in_ch, out_ch, t_kernel=3):
        super().__init__()
        self.spatial_fc = nn.Linear(in_ch, out_ch)   # simulates GCN W^(l)
        self.temporal   = nn.Conv1d(out_ch, out_ch, t_kernel, padding=t_kernel//2)
        self.norm       = nn.LayerNorm(out_ch)
        self.act        = nn.ReLU()
        self.res        = nn.Linear(in_ch, out_ch) if in_ch != out_ch else nn.Identity()

    def forward(self, x):                    # [B, T, C]
        # Spatial graph convolution
        h = self.act(self.spatial_fc(x))     # [B, T, out]
        # Temporal convolution
        h = h.permute(0,2,1)                 # [B, out, T]
        h = self.temporal(h).permute(0,2,1)  # [B, T, out]
        # Residual + norm  (Paper 1: ensures feature propagation)
        return self.norm(h + self.res(x))


class SignEncoder(nn.Module):
    """
    ST-GCN based sign encoder (Paper 1) + Transformer positional encoding.
    4 ST-GCN blocks → BiLSTM → d_model dimensional sign features S.
    """
    def __init__(self, input_dim, d_model=256, n_heads=4, n_layers=4):
        super().__init__()
        # ST-GCN blocks (from Paper 1)
        self.stgcn = nn.Sequential(
            STGCNBlock(input_dim, 256),
            STGCNBlock(256, 256),
            STGCNBlock(256, 256),
            STGCNBlock(256, d_model),
        )
        # Temporal BiLSTM for sequence-level context
        self.lstm = nn.LSTM(d_model, d_model//2, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=0.3)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(0.3)

    def forward(self, x):                   # [B, T, F]
        x = self.stgcn(x)                   # [B, T, d_model]  ST-GCN features
        x, _ = self.lstm(x)                 # [B, T, d_model]  temporal context
        return self.drop(self.norm(x))       # sign features S

---
## Phase 4 — EMF-SLT Components (Paper 2)
> **From Hu et al. (ICASSP 2024):**
> - **Vector Quantizer**: align sign and gloss in shared discrete codebook (Eq.1-2)
> - **Fusion Module**: cross-attention [S; S'] as Q, S as K,V (Eq.3-5)
> - **Multi-task mutual learning**: S2T + G2T + KL divergence (Eq.6-8)

In [5]:
class VectorQuantizer(nn.Module):
    """
    Vector Quantizer for cross-modal alignment. (Paper 2, Section 2.1)
    Aligns sign features Ŝ with gloss features Ĝ in shared discrete codebook.
    L_Align = -Σ sg[p_g] log(p_s)   — Eq(1) from Paper 2
    Uses Gumbel-Softmax for differentiable discrete selection — Eq(2) from Paper 2
    """
    def __init__(self, d_model=256, codebook_size=50, n_queries=16, tau=0.1):
        super().__init__()
        self.tau      = tau
        self.n_q      = n_queries
        self.codebook = nn.Embedding(codebook_size, d_model)

        # Cross-attention to compress variable-length S and G to N fixed queries
        self.query    = nn.Parameter(torch.randn(1, n_queries, d_model))
        self.attn_s   = nn.MultiheadAttention(d_model, 4, batch_first=True, dropout=0.1)
        self.attn_g   = nn.MultiheadAttention(d_model, 4, batch_first=True, dropout=0.1)

        # Projection to codebook logits
        self.proj_s   = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, codebook_size))
        self.proj_g   = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, codebook_size))

    def forward(self, S, G, training=True):
        """S: sign feats [B,Ls,D], G: gloss feats [B,Lg,D]
        Returns: Ŝ' (aligned sign), Ĝ, L_align"""
        B = S.shape[0]
        Q = self.query.expand(B, -1, -1)          # [B, N, D]

        # Compress to N queries via cross-attention (Q-Former style, Paper 2)
        S_hat, _ = self.attn_s(Q, S, S)           # [B, N, D]
        G_hat, _ = self.attn_g(Q, G, G)           # [B, N, D]

        # Compute codebook logits
        ls = self.proj_s(S_hat)                    # [B, N, V]
        lg = self.proj_g(G_hat)                    # [B, N, V]

        # Gumbel-Softmax  — Eq(2) from Paper 2
        p_s = F.gumbel_softmax(ls, tau=self.tau, hard=False, dim=-1)
        p_g = F.gumbel_softmax(lg, tau=self.tau, hard=False, dim=-1)

        # Alignment loss: -Σ sg[p_g] log(p_s)  — Eq(1)
        L_align = -(p_g.detach() * (p_s + 1e-8).log()).sum(-1).mean()

        # Select discrete tokens: S' = C_k where k = argmax(p_s)
        hard_idx = p_s.argmax(-1)                  # [B, N]
        S_prime  = self.codebook(hard_idx)          # [B, N, D]

        return S_prime, G_hat, L_align


class FusionModule(nn.Module):
    """
    Multi-modal Fusion. (Paper 2, Eq.3-5)
    Q = [S; S'] (concat sign + aligned sign)
    K = V = S   (sign features)
    f_M = MultiHead(Q, K, V)
    """
    def __init__(self, d_model=256, n_heads=4):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, S, S_prime):          # S:[B,Ls,D]  S':[B,N,D]
        Q = torch.cat([S, S_prime], dim=1)  # [B, Ls+N, D]  — Eq(3)
        K = V = S                            #               — Eq(4)
        fM, _ = self.cross_attn(Q, K, V)   # [B, Ls+N, D] — Eq(5)
        return self.norm(fM)

In [6]:
class TransformerDecoder(nn.Module):
    """Standard Transformer decoder — 3 layers as in Paper 2."""
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_layers=3, max_len=50, p=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos   = nn.Embedding(max_len, d_model)
        dec_layer  = nn.TransformerDecoderLayer(d_model, n_heads, dim_feedforward=512,
                                                 dropout=p, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=n_layers)
        self.head    = nn.Linear(d_model, vocab_size)
        self.drop    = nn.Dropout(p)

    def forward(self, tgt, memory, tgt_key_padding_mask=None):
        pos = torch.arange(tgt.shape[1], device=tgt.device).unsqueeze(0)
        x   = self.drop(self.embed(tgt) + self.pos(pos))
        # Causal mask
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.shape[1], device=tgt.device)
        out = self.decoder(x, memory, tgt_mask=tgt_mask,
                           tgt_key_padding_mask=tgt_key_padding_mask)
        return self.head(out)   # [B, T, V]


# ── Full EMF-SLT Model ────────────────────────────────────────────
class EMF_SLT(nn.Module):
    """
    Combined model from both papers:
    - Sign Encoder: ST-GCN (Paper 1) + CTC head
    - Gloss Encoder: Transformer encoder (Paper 2)
    - Vector Quantizer: cross-modal alignment (Paper 2)
    - Fusion Module: cross-attention (Paper 2)
    - Text Decoder: Transformer decoder × 2 tasks (Paper 2)
    """
    def __init__(self, input_dim, vocab_size, d_model=256, codebook_size=50):
        super().__init__()
        D = d_model

        # Paper 1: ST-GCN sign encoder
        self.sign_enc = SignEncoder(input_dim, d_model=D)

        # CTC head on sign encoder (for recognition, Paper 1 auxiliary)
        self.ctc_head = nn.Linear(D, vocab_size)

        # Paper 2: Gloss encoder (Transformer)
        self.gloss_embed = nn.Embedding(vocab_size, D, padding_idx=PAD_IDX)
        enc_layer = nn.TransformerEncoderLayer(D, 4, dim_feedforward=512, dropout=0.3, batch_first=True)
        self.gloss_enc = nn.TransformerEncoder(enc_layer, num_layers=3)

        # Paper 2: Vector Quantizer + Fusion
        self.vq      = VectorQuantizer(D, codebook_size=codebook_size, n_queries=16)
        self.fusion  = FusionModule(D)

        # Paper 2: Two decoders (S2T and G2T for mutual learning)
        self.decoder_s2t = TransformerDecoder(vocab_size, D)   # uses fused multi-modal
        self.decoder_g2t = TransformerDecoder(vocab_size, D)   # uses gloss features

    def forward(self, x, gloss_ids, tgt, training=True):
        """x: [B,T,F]  gloss_ids: [B,Lg]  tgt: [B,Lt]"""
        # 1. Sign encoding (ST-GCN, Paper 1)
        S = self.sign_enc(x)                          # [B, Ls, D]
        ctc_logits = self.ctc_head(S)                 # [B, Ls, V] for CTC

        # 2. Gloss encoding (Transformer, Paper 2)
        pad_mask = (gloss_ids == PAD_IDX)
        G = self.gloss_enc(self.gloss_embed(gloss_ids),
                           src_key_padding_mask=pad_mask)  # [B, Lg, D]

        # 3. Vector Quantizer alignment (Paper 2)
        S_prime, G_hat, L_align = self.vq(S, G, training)

        # 4. Fusion: fM = MultiHead([S; S'], S, S) (Paper 2 Eq.3-5)
        fM = self.fusion(S, S_prime)                  # [B, Ls+N, D]

        # 5. Decode: S2T from fused, G2T from gloss (Paper 2 Eq.6)
        tgt_in  = tgt[:, :-1]                         # teacher forcing
        tgt_pad = (tgt_in == PAD_IDX)
        logits_s2t = self.decoder_s2t(tgt_in, fM, tgt_key_padding_mask=tgt_pad)
        logits_g2t = self.decoder_g2t(tgt_in, G,  tgt_key_padding_mask=tgt_pad)

        return ctc_logits, logits_s2t, logits_g2t, L_align


model = EMF_SLT(INPUT_DIM, VOCAB_SIZE, d_model=256, codebook_size=50).to(DEVICE)
n = sum(p.numel() for p in model.parameters())
print(f'EMF-SLT Parameters: {n:,}')
print(f'Components: ST-GCN encoder | VQ (codebook=50) | Fusion | 2× Transformer decoder')

EMF-SLT Parameters: 9,492,620
Components: ST-GCN encoder | VQ (codebook=50) | Fusion | 2× Transformer decoder


---
## Phase 5 — Training with Multi-Task Losses (Paper 2, Eq.6-8)
> `L = L_NLL + α·L_KL + β·L_Align + γ·L_CTC`
> Paper 2 uses α=1, β=0.5; we add γ=0.1 for CTC auxiliary (Paper 1)

In [7]:
EPOCHS=100; LR=5e-4; PATIENCE=20
ALPHA=1.0; BETA=0.5; GAMMA=0.1   # Paper 2: α=1, β=0.5; + Paper 1 CTC

ctc_loss = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True, reduction='mean')
ce_loss  = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.2)  # Paper 2: label_smoothing=0.2
optimizer= torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.998))  # Paper 2 optimizer

def lr_fn(ep):
    warmup=10
    if ep<warmup: return (ep+1)/warmup
    return 0.5*(1+math.cos(math.pi*(ep-warmup)/max(EPOCHS-warmup,1)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)


def compute_loss(batch, train=True):
    xs, gs, ts, xl, gl = [b.to(DEVICE) for b in batch]
    ctc_logits, ls2t, lg2t, L_align = model(xs, gs, ts, training=train)

    # CTC loss (Paper 1: recognition auxiliary)
    lp  = F.log_softmax(ctc_logits, dim=-1).permute(1,0,2)  # [T,B,V]
    il  = xl.clamp(max=lp.shape[0])
    L_ctc = ctc_loss(lp, gs, il, gl)

    # NLL losses: S2T and G2T (Paper 2, Eq.6)
    tgt_out = ts[:, 1:].contiguous()  # shift right
    L_s2t = ce_loss(ls2t.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
    L_g2t = ce_loss(lg2t.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
    L_NLL = L_s2t + L_g2t

    # KL mutual learning (Paper 2, Eq.7)
    ps2t = F.softmax(ls2t.detach(), dim=-1) + 1e-8
    pg2t = F.softmax(lg2t.detach(), dim=-1) + 1e-8
    L_KL = (ps2t*(ps2t.log()-pg2t.log())).sum(-1).mean() + \
           (pg2t*(pg2t.log()-ps2t.log())).sum(-1).mean()

    # Total: L = NLL + α·KL + β·Align + γ·CTC  (Paper 2 Eq.8 + Paper 1)
    total = L_NLL + ALPHA*L_KL + BETA*L_align + GAMMA*L_ctc
    return total, L_NLL.item(), L_KL.item(), L_align.item()


def run_epoch(loader, train=True):
    model.train(train)
    total_l, nll_l, n = 0.0, 0.0, 0
    for batch in loader:
        if train: optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            loss, nll, kl, align = compute_loss(batch, train)
        if train and not (torch.isnan(loss) or torch.isinf(loss)):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if not torch.isnan(loss): total_l+=loss.item(); nll_l+=nll; n+=1
    return total_l/max(n,1), nll_l/max(n,1)


train_h, val_h = [], []
best_val, no_imp = float('inf'), 0
print(f'Training EMF-SLT | {EPOCHS}ep | L=NLL+{ALPHA}·KL+{BETA}·Align+{GAMMA}·CTC')
print(f'Label smoothing=0.2 | Adam(β1=0.9,β2=0.998) | paper2 settings')
print('─'*75)

for ep in range(1, EPOCHS+1):
    tl, tnll = run_epoch(train_loader, True)
    vl, vnll = run_epoch(val_loader,   False)
    scheduler.step()
    train_h.append(tl); val_h.append(vl)
    lr_now = optimizer.param_groups[0]['lr']
    star = ''
    if vl < best_val:
        best_val=vl; no_imp=0
        torch.save({'epoch':ep,'model':model.state_dict(),'vocab':vocab,'val_loss':vl}, CKPT_PATH)
        star = '  ✅'
    else: no_imp+=1
    print(f'Ep {ep:3d}/{EPOCHS}  total={tl:.4f}  nll={tnll:.4f}  val={vl:.4f}  lr={lr_now:.2e}{star}')
    if no_imp>=PATIENCE: print(f'Early stop ep {ep}'); break

print(f'Best val: {best_val:.4f}')

Training EMF-SLT | 100ep | L=NLL+1.0·KL+0.5·Align+0.1·CTC
Label smoothing=0.2 | Adam(β1=0.9,β2=0.998) | paper2 settings
───────────────────────────────────────────────────────────────────────────


C:\Users\harsh\AppData\Roaming\Python\Python310\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Ep   1/100  total=22.1005  nll=9.3887  val=0.0000  lr=1.00e-04  ✅
Ep   2/100  total=17.8531  nll=7.9786  val=0.0000  lr=1.50e-04
Ep   3/100  total=17.1573  nll=7.0638  val=0.0000  lr=2.00e-04
Ep   4/100  total=16.4748  nll=6.2164  val=0.0000  lr=2.50e-04
Ep   5/100  total=16.0159  nll=5.5630  val=0.0000  lr=3.00e-04
Ep   6/100  total=15.6492  nll=5.0058  val=0.0000  lr=3.50e-04
Ep   7/100  total=15.5618  nll=4.6772  val=0.0000  lr=4.00e-04
Ep   8/100  total=15.3580  nll=4.4002  val=0.0000  lr=4.50e-04
Ep   9/100  total=15.1958  nll=4.2064  val=0.0000  lr=5.00e-04
Ep  10/100  total=15.0603  nll=4.1158  val=0.0000  lr=5.00e-04
Ep  11/100  total=14.9300  nll=4.0529  val=0.0000  lr=5.00e-04
Ep  12/100  total=14.8645  nll=4.0011  val=0.0000  lr=4.99e-04
Ep  13/100  total=14.7103  nll=3.9537  val=0.0000  lr=4.99e-04
Ep  14/100  total=14.7672  nll=3.9339  val=0.0000  lr=4.98e-04
Ep  15/100  total=14.6386  nll=3.9176  val=0.0000  lr=4.96e-04
Ep  16/100  total=14.6991  nll=3.9013  val=0.0000  l

In [ ]:
fig,ax=plt.subplots(figsize=(12,5))
ax.plot(train_h,color='#3fb950',lw=2,label='Train Total Loss')
ax.plot(val_h,  color='#f85149',lw=2,label='Val Total Loss')
ax.set_title('EMF-SLT Training (ST-GCN + VQ + Fusion + KL)'); ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(BASE,'emf_slt_training.png'),dpi=150); plt.show()

---
## Phase 6 — Evaluation: BLEU + ROUGE (Paper 2 metrics)

In [ ]:
ckpt=torch.load(CKPT_PATH,map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print(f'Loaded: epoch={ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}')


def beam_decode(x_feat, gloss_ids, max_len=25, beam_size=5):
    """Greedy decode from S2T path (fused multi-modal features)."""
    model.eval()
    with torch.no_grad():
        x  = torch.tensor(x_feat).unsqueeze(0).to(DEVICE)
        gs = torch.tensor(gloss_ids).unsqueeze(0).to(DEVICE)
        S  = model.sign_enc(x)
        pad_mask = torch.zeros(1,gs.shape[1],dtype=torch.bool,device=DEVICE)
        G  = model.gloss_enc(model.gloss_embed(gs), src_key_padding_mask=pad_mask)
        S_prime, G_hat, _ = model.vq(S, G, training=False)
        fM = model.fusion(S, S_prime)

        generated = [BOS_IDX]
        for _ in range(max_len):
            tgt = torch.tensor([generated],dtype=torch.long,device=DEVICE)
            out = model.decoder_s2t(tgt, fM)
            nxt = out[0,-1].argmax().item()
            if nxt == EOS_IDX: break
            generated.append(nxt)
    return [idx2tok.get(i,'<unk>') for i in generated[1:]]


def bleu_n(ref, hyp, n):
    """Compute BLEU-n for a single pair."""
    if len(hyp)<n: return 0.0
    ref_ng = set(zip(*[ref[i:] for i in range(n)]))
    hyp_ng = [tuple(hyp[i:i+n]) for i in range(len(hyp)-n+1)]
    if not hyp_ng: return 0.0
    matches = sum(1 for ng in hyp_ng if ng in ref_ng)
    return matches / len(hyp_ng)

def rouge_l(ref, hyp):
    """ROUGE-L (LCS-based)."""
    m,n=len(ref),len(hyp)
    dp=np.zeros((m+1,n+1))
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j]=dp[i-1][j-1]+1 if ref[i-1]==hyp[j-1] else max(dp[i-1][j],dp[i][j-1])
    lcs=dp[m][n]
    p=lcs/max(n,1); r=lcs/max(m,1)
    return 2*p*r/max(p+r,1e-8)


bleu_scores={1:[],2:[],3:[],4:[]}; rouge_scores=[]; examples=[]

for np_, gloss_ids, tgt_ids, label in val_ds.samples:
    x_feat = np.nan_to_num(np.load(np_).astype(np.float32))
    pred = beam_decode(x_feat, gloss_ids)
    ref  = tokenize(label)
    for n in [1,2,3,4]: bleu_scores[n].append(bleu_n(ref, pred, n))
    rouge_scores.append(rouge_l(ref, pred))
    if len(examples)<8: examples.append((label,' '.join(pred)))

print(f'\n{"═"*60}')
for n in [1,2,3,4]:
    print(f'  BLEU-{n}  : {np.mean(bleu_scores[n])*100:.2f}%')
print(f'  ROUGE-L : {np.mean(rouge_scores)*100:.2f}%')
print(f'{"═"*60}\n')
print(f'{"REFERENCE":42s}  PREDICTION')
print('─'*90)
for ref,hyp in examples: print(f'{ref[:42]:42s}  {hyp[:42]}')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle('EMF-SLT Evaluation (BLEU & ROUGE)',color='#58a6ff',fontsize=14)
ns=['BLEU-1','BLEU-2','BLEU-3','BLEU-4']
bv=[np.mean(bleu_scores[i])*100 for i in [1,2,3,4]]
bars=axes[0].bar(ns,bv,color=['#58a6ff','#3fb950','#ffa657','#f85149'],alpha=0.85,width=0.5)
for bar,v in zip(bars,bv): axes[0].text(bar.get_x()+bar.get_width()/2,v+0.5,f'{v:.1f}%',ha='center',color='white',fontsize=11)
axes[0].set_ylim(0,max(bv)*1.3+5); axes[0].set_title('BLEU Scores'); axes[0].grid(True,alpha=0.3)
rv=np.mean(rouge_scores)*100
axes[1].bar(['ROUGE-L'],[rv],color='#d2a8ff',alpha=0.85,width=0.4)
axes[1].text(0,rv+0.5,f'{rv:.1f}%',ha='center',color='white',fontsize=14,fontweight='bold')
axes[1].set_ylim(0,max(rv*1.3,20)); axes[1].set_title('ROUGE-L Score'); axes[1].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(BASE,'emf_slt_eval.png'),dpi=150); plt.show()

---
## Architecture Reference

| Component | Paper | Implementation |
|---|---|---|
| **MediaPipe keypoints** | Paper 1 §IV-A | 225D (pose+hands) |
| **ST-GCN encoder** | Paper 1 §IV-D, Eq(1)(2) | 4× STGCNBlock + BiLSTM |
| **CTC auxiliary loss** | Paper 1 | On sign encoder output |
| **Vector Quantizer** | Paper 2 §2.1, Eq(1)(2) | Gumbel-Softmax, codebook=50 |
| **Fusion Module** | Paper 2 §2.1, Eq(3-5) | Cross-attention [S;S']→S |
| **S2T + G2T decoders** | Paper 2 §2.2, Eq(6) | 2× Transformer decoder |
| **KL mutual learning** | Paper 2 §2.2, Eq(7) | Bidirectional KL |
| **Total loss** | Paper 2, Eq(8) | NLL + 1.0·KL + 0.5·Align + 0.1·CTC |
| **Evaluation** | Paper 2 §3.1 | BLEU-1/2/3/4 + ROUGE-L |